In [12]:
import asyncio
from dvsportal import DVSPortal
from dateutil.relativedelta import relativedelta, TU
from datetime import datetime, timedelta

# Specify connection

In [13]:
from credentials import *

In [14]:
api_host = "vergunningen.parkerendelft.com"

In [15]:
portal = DVSPortal(api_host=api_host, identifier=identifier, password=password)

/var/folders/vw/mxq27d3973591vwhpnkhn28r0000gn/T/ipykernel_43711/2827430348.py:1: UserWarning: DVSPortal instance was not properly closed. Call `await close()` or use `async with DVSPortal()`.
  portal = DVSPortal(api_host=api_host, identifier=identifier, password=password)
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x10572eb50>


# Checking stuff

In [16]:
await portal.update()

In [17]:
portal.balance

5903

In [18]:
portal.active_reservations

{}

# Making a a series of reservations

Oh, awesome! The web interface places a limit in the UI on how many weeks in advance you can make a booking (typically 8 weeks). But via the API, there is no limit!!!!!

Oh, no, that is not true: it does not give an error. And via the API, the latest reservation is 25 aug 2026 right now. But the web interface shows the last booking as 4 aug 2026. 

So, either they are just hidden in the web interface, or the error is not being triggered which makes the code below thing the creation of the reservation was successful.

In [25]:
num_weeks_ahead = range(1,20)

for n in num_weeks_ahead:
    start_datetime = datetime.now() + relativedelta(weekday=TU(n), hour=9, minute=0, second=0, microsecond=0)
    start_datetime
    
    end_datetime = start_datetime + timedelta(hours=4)
    end_datetime
    
    result = await portal.create_reservation(license_plate_value=plate, date_from=start_datetime, date_until=end_datetime)

# Checking resevations

This does not seem to work properly, I have now 16 reservations but this shows only one? (The most recent one, for Aug 25, which does not show up on the web interface...). We'll I'm happy I don't have to do as much clicking at least. 

In [26]:
await portal.update()
portal.active_reservations

{'G237RP': {'reservation_id': 7138113,
  'valid_from': '2027-01-05T09:00:00',
  'valid_until': '2027-01-05T13:00:00',
  'license_plate': 'G237RP',
  'units': 60,
  'cost': 0.09675}}